In [1]:
!pip install prophet pandas

In [37]:
import pandas as pd
import random as rd

STARTING_BALANCE = 2000.00
PAYCHECK_DAYS = (1, 15)
PAYCHECK_AMOUNT = 900.00
BILLS_DAY = 1
BILLS_AMOUNT = 850.00
WEEKDAYS_EXPENSE_LIMITS = (5.00, 40.00)
WEEKEND_EXPENSE_LIMITS = (20.00, 100.00)

date_range = pd.date_range(start='2023-01-01', periods=1000, freq='D')
df = pd.DataFrame(date_range, columns=['ds'])
df['y'] = 0.00

current_balance = STARTING_BALANCE
spending_probabilities = [0.5, 0.2, 0.4, 0.5, 0.75, 0.9, 0.6]
for date in date_range:
    if date.day in PAYCHECK_DAYS:
        current_balance += PAYCHECK_AMOUNT
    if date.day == BILLS_DAY:
        current_balance -= BILLS_AMOUNT

    day_of_week = date.weekday()
    if rd.random() < spending_probabilities[day_of_week]:
        if day_of_week < 4:
            current_balance -= rd.uniform(WEEKDAYS_EXPENSE_LIMITS[0], WEEKDAYS_EXPENSE_LIMITS[1])
        else:
            current_balance -= rd.uniform(WEEKEND_EXPENSE_LIMITS[0], WEEKEND_EXPENSE_LIMITS[1])

    df.loc[df['ds'] == date, 'y'] = round(current_balance, 2) # Round to two decimal places

display(df)

,ds,y
0,2023-01-01,2011.42
1,2023-01-02,2011.42
2,2023-01-03,2011.42
3,2023-01-04,2011.42
4,2023-01-05,1977.05
...,...,...
995,2025-09-22,8779.15
996,2025-09-23,8763.70
997,2025-09-24,8741.86
998,2025-09-25,8704.96


In [38]:
from prophet import Prophet

model = Prophet(daily_seasonality=False, weekly_seasonality=True, yearly_seasonality=False)
model.add_seasonality(name='monthly', period=30.5, fourier_order=5)
model.add_seasonality(name='paycheck', period=14, fourier_order=5)
model.fit(df)

future = model.make_future_dataframe(periods=30)
forecast = model.predict(future)

display(forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail(30))

DEBUG:cmdstanpy:input tempfile: /tmp/tmpgkd32vpo/bb30ze5v.json
DEBUG:cmdstanpy:input tempfile: /tmp/tmpgkd32vpo/j2sjoyq3.json
DEBUG:cmdstanpy:idx 0
DEBUG:cmdstanpy:running CmdStan, num_threads: None
DEBUG:cmdstanpy:CmdStan args: ['/usr/local/lib/python3.12/dist-packages/prophet/stan_model/prophet_model.bin', 'random', 'seed=74756', 'data', 'file=/tmp/tmpgkd32vpo/bb30ze5v.json', 'init=/tmp/tmpgkd32vpo/j2sjoyq3.json', 'output', 'file=/tmp/tmpgkd32vpo/prophet_model5zs0q8tz/prophet_model-20251018211414.csv', 'method=optimize', 'algorithm=lbfgs', 'iter=10000']
21:14:14 - cmdstanpy - INFO - Chain [1] start processing
INFO:cmdstanpy:Chain [1] start processing
21:14:14 - cmdstanpy - INFO - Chain [1] done processing
INFO:cmdstanpy:Chain [1] done processing


,ds,yhat,yhat_lower,yhat_upper
1000,2025-09-27,8858.638047,8706.357779,9009.011965
1001,2025-09-28,8846.119933,8694.794382,9022.682181
1002,2025-09-29,8849.596239,8689.399480,9026.881030
1003,2025-09-30,8833.707127,8671.788419,8993.224037
1004,2025-10-01,8815.977470,8664.423441,8984.118113
1005,2025-10-02,8815.012624,8652.978803,8986.817545
1006,2025-10-03,8794.814684,8637.076875,8969.184632
1007,2025-10-04,8756.843109,8598.234311,8924.378024
1008,2025-10-05,8714.323806,8544.804381,8882.801752
1009,2025-10-06,8669.395740,8508.210310,8832.463524
